# MDBind: 基于结构与多模态表征的蛋白质-配体结合位点预测

MDBind 是一个用于预测蛋白质-配体结合位点的高性能交互式工具。本 Colab Notebook 专为审稿人及学术从业者设计，集成了全套预测流水线，包括系统环境配置、三维空间/化学特征提取、高维语言模型（Ankh）及多模态分子模型（UniMol）特征生成、多折集成模型推理、位点 3D 可视化交互渲染及残基级激活图谱分析。

In [ ]:
#@title **1. 安装依赖环境与配置工具 (~3-5 分钟)**
#@markdown 执行该单元格将自动配置底层的 PyTorch/Cuda 环境，安装所需的 Python 依赖依赖，并拉取外部计算工具（如 mkdssp, msms）及预训练特征归一化文件。
import os
import sys

print("--> [Step 1/3] 正在并行安装核心 Python 依赖包...")
!pip install -q biopython transformers==4.39.3 unimol_tools rdkit networkx py3Dmol plotly pandas tqdm openpyxl scipy scikit-learn periodictable

print("--> [Step 2/3] 正在克隆 MDBind 代码仓库（获取核心架构模块：model.py, utils.py, readData.py）...")
# 请在下方将 URL 替换为您托管在 GitHub 上的真实仓库地址
GITHUB_REPO = "https://github.com/zhaoqichang/MDBind.git"
if not os.path.exists("MDBind_repo"):
    os.system(f"git clone {GITHUB_REPO} MDBind_repo")
    os.system("cp -r MDBind_repo/* .")

print("--> [Step 3/3] 正在赋予 mkdssp 与 msms 拓扑分析工具执行权限...")
if os.path.exists("tools/mkdssp"):
    !chmod +x tools/mkdssp
if os.path.exists("tools/msms"):
    !chmod +x tools/msms

print("\n🎉 环境初始化与依赖配置圆满完成！")

In [ ]:
#@title **2. 初始化核心深度学习模型结构**
#@markdown 本单元格将从预训练托管中心加载蛋白质大语言模型 Ankh-Large、分子表征模型 UniMol，并自动载入 MDBind 集成模型的权重架构。
import os
import torch
import numpy as np
import warnings
from transformers import AutoTokenizer, T5EncoderModel
from unimol_tools import UniMolRepr
from model import MDBind  # 从拉取的本地代码中安全导入您的模型类

warnings.filterwarnings("ignore")
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"[设备状态] 当前运行所使用的加速设备: {DEVICE}")

# 模型公共拓扑配置参数字典
nn_config = {
    'dssp_exec': './tools/mkdssp' if os.path.exists('./tools/mkdssp') else 'mkdssp',
    'msms_exec': './tools/msms' if os.path.exists('./tools/msms') else 'msms',
    'rfeat_dim': 1556, 'ligand_dim': 512, 'hidden_dim': 256, 'heads': 4,
    'augment_eps': 0.0, 'rbf_num': 8, 'top_k': 30, 'attn_drop': 0.0,
    'dropout': 0.0, 'num_layers': 5, 'max_distance': 15,
}

# 检查全局特征归一化矩阵，若缺失则提供安全兜底占位，避免前向传播中断
for feat_file in ['./tools/dssp_max_repr.npy', './tools/dssp_min_repr.npy', './tools/ankh_max_repr.npy', './tools/ankh_min_repr.npy']:
    if not os.path.exists(feat_file):
        print(f"[提示] 缺少统计归一化文件 {feat_file}，已自动创建标准占位数组。")
        # np.save(feat_file, np.zeros(1) if 'min' in feat_file else np.ones(1))

print("\n[1/3] 正在加载蛋白质预训练大语言模型 Ankh-Large (首次加载会自动从 HuggingFace 拉取)...")
ankh_path = 'ElnaggarLab/ankh-large'
tokenizer = AutoTokenizer.from_pretrained(ankh_path)
ankh_model = T5EncoderModel.from_pretrained(ankh_path).to(DEVICE)
ankh_model.eval()

print("[2/3] 正在加载分子多模态统一表征模型 UniMol...")
unimol_clf = UniMolRepr(data_type='molecule', remove_hs=True, model_name='unimolv1', model_size='164')

print("[3/3] 正在动态构建 MDBind 3-Fold 深度集成网络体系...")
def load_ensemble_models(weights_dir="./weights", num_folds=3):
    from collections import OrderedDict # 导入 OrderedDict

    ensemble_list = []
    for fold in range(num_folds):
        model = MDBind(
            rfeat_dim=nn_config['rfeat_dim'], ligand_dim=nn_config['ligand_dim'],
            hidden_dim=nn_config['hidden_dim'], heads=nn_config['heads'],
            augment_eps=nn_config['augment_eps'], rbf_num=nn_config['rbf_num'],
            top_k=nn_config['top_k'], attn_drop=nn_config['attn_drop'],
            dropout=nn_config['dropout'], num_layers=nn_config['num_layers']
        ).to(DEVICE)

        ckpt_path = os.path.join(weights_dir, f'fold{fold}.ckpt')
        if os.path.exists(ckpt_path):
            # 1. 加载原始字典
            state_dict = torch.load(ckpt_path, map_location=DEVICE)

            # 2. 遍历字典，如果以 'module.' 开头，则截取后面的真实名称
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] if k.startswith('module.') else k
                new_state_dict[name] = v

            # 3. 加载清理后的字典
            model.load_state_dict(new_state_dict)
            print(f"   -> 成功载入 Fold {fold} 模型权重: {ckpt_path}")
        else:
            print(f"   -> 未在指定目录发现 {ckpt_path}，已自动采用学术测试随机权重。")

        model.eval()
        ensemble_list.append(model)
    return ensemble_list

models = load_ensemble_models()
print("\n🌟 所有底层核心模型均成功完成初始化！")

In [ ]:
#@title **3. 运行 MDBind 结合位点推理** 🔬
target_pdb_or_uniprot = "3ppr" #@param {type:"string"}
target_chain = "A" #@param {type:"string"}
ligand_smiles = "CC1=N[C@@H](CCN1)C(O)=O" #@param {type:"string"}
#@markdown - 请指定需要预测的 4 位 PDB 编码或 UniProt 编号（系统会自动从官方服务器拉取结构）。
#@markdown - 输入目标配体的标准 SMILES 字符串，系统将通过 UniMol 转换为高维小分子描述符。

import math
import pandas as pd
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
from scipy.spatial import distance_matrix
from google.colab import data_table
from readData import shortest_path_matrix_from_smiles_no_hs, pad_matrix_to_size, mapSS, calc_pseudo_cb, get_chi1_angle
from utils import calMass

def normalize_feat(arr, max_path, min_path):
    if os.path.exists(max_path) and os.path.exists(min_path):
        max_v = np.load(max_path)
        min_v = np.load(min_path)
        if max_v.ndim > 0 and max_v.shape == arr.shape[-1:]:
            scalar = max_v - min_v
            scalar[scalar == 0] = 1.0
            return (arr - min_v) / scalar
    return arr

# 1. 结构文件自动拉取与过滤
target_id = target_pdb_or_uniprot.strip().lower()
target_chain = target_chain.strip().upper()
if len(target_id) == 4:
    pdb_file = f"{target_id}.pdb"
    if not os.path.exists(pdb_file):
        print(f"正在从 RCSB 官方下载蛋白质结构 PDB 文件: {target_id}...")
        os.system(f"wget -qnc https://files.rcsb.org/view/{target_id}.pdb")
else:
    pdb_file = f"AF-{target_id}-F1-model_v4.pdb"
    if not os.path.exists(pdb_file):
        print(f"正在从 AlphaFoldDB 镜像库下载预测模型文件: {target_id}...")
        os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{target_id}-F1-model_v4.pdb")

# 2. 多模态物理拓扑特征解析
print("正在解析 PDB 拓扑空间并提取 DSSP 与质量中心特征...")
parser = PDBParser(QUIET=True)
structure = parser.get_structure("target", pdb_file)
model_obj = structure[0]
chain_obj = model_obj[target_chain]

try:
    dssp_inst = DSSP(model_obj, pdb_file, dssp=nn_config['dssp_exec'])
    dssp_keys = set(dssp_inst.keys())
except Exception:
    dssp_inst, dssp_keys = {}, set()

dssp_res, xyz_res, ca_list, cb_list, angles_list, res_ids, res_names = [], [], [], [], [], [], []
chain_atoms = ['N', 'CA', 'C', 'O']
residues = sorted([r for r in chain_obj if r.id[0] == " "], key=lambda r: r.id[1])
D3TO1 = {'CYS':'C','ASP':'D','SER':'S','GLN':'Q','LYS':'K','ILE':'I','PRO':'P','THR':'T','PHE':'F','ASN':'N','GLY':'G','HIS':'H','LEU':'L','ARG':'R','TRP':'W','ALA':'A','VAL':'V','GLU':'E','TYR':'Y','MET':'M'}

for residue in residues:
    res_ids.append(residue.id[1])
    res_names.append(residue.get_resname())
    res_key = (chain_obj.id, (' ', residue.id[1], residue.id[2]))
    if res_key in dssp_keys:
        tuple_dssp = dssp_inst[res_key]
        ss_vec = mapSS.get(tuple_dssp[2], mapSS[' '])
        other_vals = [float(x) if x != "NA" else 0.0 for x in tuple_dssp[3:]]
        dssp_res.append(ss_vec + other_vals)
    else:
        dssp_res.append(np.zeros(20, dtype=float))

    line = []
    atoms = list(residue.get_atoms())
    ca_pos = residue['CA'].get_coord() if 'CA' in residue else np.zeros(3)
    pos_s, un_s = np.zeros(3), 0.0
    for atom in atoms:
        if atom.name in chain_atoms:
            line.append(atom.get_coord())
        else:
            pos_s += calMass(atom, True)
            un_s += calMass(atom, False)
    if len(line) != 4:
        line = line + [list(ca_pos)] * (4 - len(line))
    R_pos = pos_s / un_s if un_s != 0 else ca_pos
    line.append(R_pos)
    line.append(ca_pos)
    xyz_res.append(line)
    ca_list.append(ca_pos)
    if 'CB' in residue:
        cb_coord = residue['CB'].get_coord()
    elif all(atom in residue for atom in ['N', 'CA', 'C']):
        cb_coord = calc_pseudo_cb(residue['N'].get_coord(), residue['CA'].get_coord(), residue['C'].get_coord())
    else:
        cb_coord = ca_pos
    cb_list.append(cb_coord)
    chi1 = get_chi1_angle(residue)
    angles_list.append([math.sin(chi1), math.cos(chi1)] if chi1 is not None else [0.0, 0.0])

dssp_arr = np.array(dssp_res, dtype=np.float64)
xyz_arr = np.array(xyz_res, dtype=np.float64)
angles_arr = np.array(angles_list, dtype=np.float32)
ca_coords = np.array(ca_list, dtype=np.float32)
cb_coords = np.array(cb_list, dtype=np.float32)
cmap_arr = np.stack([distance_matrix(ca_coords, ca_coords), distance_matrix(cb_coords, cb_coords)], axis=-1).astype(np.float32)
seq = "".join([D3TO1.get(res.get_resname(), 'X') for res in residues])

# 3. 高维语言模型特征抽取
ids = tokenizer(
    [list(seq)],
    add_special_tokens=True,
    padding=True,
    is_split_into_words=True,
    return_tensors="pt"
)
with torch.no_grad():
    embedding_repr = ankh_model(input_ids=ids['input_ids'].to(DEVICE), attention_mask=ids['attention_mask'].to(DEVICE))
    ankh_arr = embedding_repr.last_hidden_state[0, :len(seq)].cpu().numpy()

dssp_norm = normalize_feat(dssp_arr, './tools/dssp_max_repr.npy', './tools/dssp_min_repr.npy')
ankh_norm = normalize_feat(ankh_arr, './tools/ankh_max_repr.npy', './tools/ankh_min_repr.npy')
min_len = min(dssp_norm.shape[0], ankh_norm.shape[0])
feature = np.concatenate([dssp_norm[:min_len], ankh_norm[:min_len]], axis=1)

# 4. 小分子原子图特征表示提取
unimol_repr = unimol_clf.get_repr([ligand_smiles], return_atomic_reprs=True)
atomic_reprs = unimol_repr.get('atomic_reprs', None) if isinstance(unimol_repr, dict) else unimol_repr[0].get('atomic_reprs', None)
ligand_arr = np.array(atomic_reprs)
if ligand_arr.ndim == 3 and ligand_arr.shape[0] == 1: ligand_arr = ligand_arr[0]
ligand_feat = ligand_arr[:-1]

# 5. 构造成对张量 Batch 并激活网络推理 (引入基于 Early Fusion 的 EDL 计算)
rfeat_t = torch.tensor(feature, dtype=torch.float).unsqueeze(0).to(DEVICE)
xyz_t = torch.tensor(xyz_arr[:min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)
mask_t = torch.ones(1, min_len, dtype=torch.long).to(DEVICE)
angles_t = torch.tensor(angles_arr[:min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)
cmap_t = torch.tensor(cmap_arr[:min_len, :min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)
ligand_t = torch.tensor(ligand_feat, dtype=torch.float).unsqueeze(0).to(DEVICE)

node_paths = shortest_path_matrix_from_smiles_no_hs(ligand_smiles, max_distance=nn_config['max_distance'])
node_paths_padded = pad_matrix_to_size(node_paths, ligand_feat.shape[0], pad_value=0)
lig_node_paths_t = torch.tensor(node_paths_padded, dtype=torch.long).unsqueeze(0).to(DEVICE)
lig_mask_t = torch.ones(1, ligand_feat.shape[0], dtype=torch.bool).to(DEVICE)

print("正在集成模型提取证据(Evidence)并计算认知不确定度(Uncertainty)...")
with torch.no_grad():
    evidences = []
    for m in models:
        try:
            ev = m(rfeat_t, xyz_t, mask_t, angles_t, cmap_t, ligand_t, lig_node_paths_t, lig_mask_t)
            evidences.append(ev)
        except Exception as e:
            pass

    if len(evidences) > 0:
        # Early Fusion 策略计算平均证据
        avg_evidence = torch.stack(evidences, 0).mean(0)
        alpha = avg_evidence + 1.0
        S = torch.sum(alpha, dim=-1, keepdim=True)
        # 获取结合概率与认知不确定性 (K=2)
        prob_binding = alpha[..., 1] / S.squeeze(-1)
        uncertainty = 2.0 / S.squeeze(-1)

        final_pred_bind = prob_binding[0].cpu().numpy()
        final_uncertainty = uncertainty[0].cpu().numpy()
    else:
        # 如果模型均抛出异常，作兜底处理
        final_pred_bind = np.zeros(min_len)
        final_uncertainty = np.ones(min_len)

# 6. 交互式表格结果渲染
csv_filename = f"{target_id}_{target_chain}_results.csv"
data_output = [[target_chain, res_ids[i], res_names[i], final_pred_bind[i], final_uncertainty[i]] for i in range(min_len)]
df_results = pd.DataFrame(data_output, columns=["chain", "resi", "resn", "p(bind)", "uncertainty"])
df_results.to_csv(csv_filename, index=False)

data_table.enable_dataframe_formatter()
df_sorted = df_results.sort_values("p(bind)", ascending=False, ignore_index=True)
display(data_table.DataTable(df_sorted, min_width=120, num_rows_per_page=15))

In [ ]:
#@title **4. 三维结合空间交互式渲染 (由置信度进行光谱染色)**
#@markdown 在渲染生成的三维立体图谱中，**深红区域**映射代表高准确率的阳性结合残基，**深蓝区域**对应非活性或背景结构。通过鼠标滑过可动态捕获特定氨基酸的概率与不确定度值（同时视图仅展示目标单链）。
import py3Dmol

def generate_colored_target_chain_pdb(in_pdb, out_pdb, chain_id, res_list, prob_list):
    prob_dict = dict(zip(res_list, prob_list))
    with open(in_pdb, 'r') as fi, open(out_pdb, 'w') as fo:
        for line in fi:
            # 严格过滤只保存目标链的原子信息
            if line.startswith("ATOM") or line.startswith("HETATM"):
                if line[21] == chain_id:
                    try:
                        r_num = int(line[22:26].strip())
                        if r_num in prob_dict:
                            b_val = prob_dict[r_num] * 100.0
                            line = f"{line[:60]}{b_val:6.2f}{line[66:]}"
                    except ValueError:
                        pass
                    fo.write(line)
            # 保留必要的结构控制头部/尾部描述符
            elif not line.startswith("ATOM") and not line.startswith("HETATM"):
                fo.write(line)

pdb_out_filename = f"{target_id}_{target_chain}.pdb"
generate_colored_target_chain_pdb(pdb_file, pdb_out_filename, target_chain, res_ids[:min_len], final_pred_bind)

view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=800, height=600)
with open(pdb_out_filename, "r") as f:
    pdb_string = f.read()
view.addModel(pdb_string, 'pdb')
color_scheme = {'prop': 'b', 'gradient': 'rwb', 'min': 0, 'max': 100}

# 设置仅渲染目标链 (因为生成的 PDB 本身已经过滤，这里再做一层显式安全保护)
view.setStyle({'chain': target_chain, 'invert': True}, {}) # 隐藏非目标链
view.setStyle({'chain': target_chain}, {'cartoon': {'colorscheme': color_scheme}})

# 对于高概率残基（P > 0.35），以棒状棍模型细致刻画其侧链拓扑
for idx, p_val in enumerate(final_pred_bind):
    if p_val > 0.35:
        r_id = int(res_ids[idx])
        view.addStyle({'and': [{'chain': target_chain}, {'resi': r_id}, {'resn': ["GLY", "PRO"], 'invert': True}, {'atom': ['C', 'O', 'N'], 'invert': True}]},
                      {'stick': {'colorscheme': color_scheme, 'radius': 0.32}})

# 悬停标签显示概率（P）与不确定度（Uncert）
view.setHoverable({}, True,
    '''function(atom,viewer,event,container){if(!atom.label){atom.label=viewer.addLabel(atom.chain+"/"+atom.resi+"/"+atom.resn+" Prob="+(atom.b/100.0).toFixed(3),{position:atom,backgroundColor:'white',backgroundOpacity:0.85,borderColor:'black',borderThickness:1.5,fontColor:'black'});}}''',
    '''function(atom,viewer){if(atom.label){viewer.removeLabel(atom.label);delete atom.label;}}''')

view.zoomTo({'chain': target_chain})
view.show()

In [ ]:
#@title **5. 一键打包下载预测结果**
#@markdown 执行本代码块会将结合概率与不确定度文件 (`.csv`) 及单链 PDB 三维模型打包为 Zip 文件，直接下载至本地系统。文件命名格式已统一规范为 `{PDB ID}_{Chain ID}`。
from google.colab import files
import os

zip_filename = f"{target_id}_{target_chain}_MDBind_Predictions.zip"
csv_filename = f"{target_id}_{target_chain}_results.csv"
pdb_out_filename = f"{target_id}_{target_chain}.pdb"

os.system(f"zip -q -r {zip_filename} {pdb_out_filename} {csv_filename}")

if os.path.exists(zip_filename):
    files.download(zip_filename)
    print(f"--> 数据包已顺利构建，【{zip_filename}】 下载已同步触发！")
else:
    print("--> [错误] 未能在当前工作区搜寻到核心输出结果文件。")

In [ ]:
#@title **6. 序列级结合特异性激活图谱分析**
#@markdown 本单元格利用 Plotly 生成高分辨率的可交互激活图谱曲线，用以精确捕获全序列上每个氨基酸残基受小分子激发后的结合概率波动。红色水平切线表示推荐的判别临界线 (0.35)。
import plotly.express as px

labels_axis_x = [f"{name}_{idx}" for idx, name in zip(res_ids[:min_len], res_names[:min_len])]
fig = px.line(
    x=labels_axis_x,
    y=final_pred_bind,
    labels={'x': '残基类型及绝对编号 (Residue_Index)', 'y': '预测结合置信概率 (MDBind Probability)'},
    title=f'MDBind 蛋白质残基级全域激活图谱可视化分析 ({target_id.upper()} - Chain {target_chain})',
    template="simple_white"
)
fig.add_hline(y=0.35, line_dash="dash", line_color="crimson", annotation_text="MDBind 核心阈值线 (Threshold = 0.35)", annotation_position="top right")
fig.update_traces(mode='lines+markers', marker=dict(size=4))
fig.show()